In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import utils

In [2]:
# 读取 Excel 文件
df = pd.read_excel("./Rainfall data.xlsx")

# 设置列名并处理缺失值
df.columns = ["Time", "Rainfall(mm)", "Cumulative rainfall(mm)"]
df.dropna(subset=["Time"], inplace=True)

# 将 Time 列转换为 datetime 类型，并转换为 UTC 时间（减去 8 小时）
df["Time"] = pd.to_datetime(df["Time"])
df["UTC_Time"] = df["Time"] - pd.Timedelta(hours=8)

# 按照 UTC 时间排序
df.sort_values(by="UTC_Time", inplace=True)
# 筛选从 8 月 16 日开始的数据（UTC 时间）
start_date = pd.Timestamp("2023-08-16")
df_filtered = df[df["UTC_Time"] >= start_date].copy()


In [3]:
df_filtered

,Time,Rainfall(mm),Cumulative rainfall(mm),UTC_Time
96,2023-08-16 08:03:29,0.0,0.0,2023-08-16 00:03:29
97,2023-08-16 08:08:29,0.0,0.0,2023-08-16 00:08:29
98,2023-08-16 08:13:29,0.0,0.0,2023-08-16 00:13:29
99,2023-08-16 08:18:29,0.0,0.0,2023-08-16 00:18:29
100,2023-08-16 08:23:29,0.0,0.0,2023-08-16 00:23:29
...,...,...,...,...
8616,2023-09-14 23:39:18,0.0,0.0,2023-09-14 15:39:18
8617,2023-09-14 23:44:18,0.0,0.0,2023-09-14 15:44:18
8618,2023-09-14 23:49:18,0.0,0.0,2023-09-14 15:49:18
8619,2023-09-14 23:54:18,0.0,0.0,2023-09-14 15:54:18


In [4]:
start_date = pd.Timestamp('2023-08-14 00:00:00')
end_date = pd.Timestamp('2023-09-19 00:00:00')
# 转换时间戳到样本索引 (假设采样率为250 Hz)
sample_rate = 250
# 确保 UTC_Time 是 datetime 类型
df_filtered['UTC_Time'] = pd.to_datetime(df_filtered['UTC_Time'])
# 设置 UTC_Time 为索引
df_filtered.set_index('UTC_Time', inplace=True)
# 计算 UTC_Time 到样本索引的映射
df_filtered['idx'] = ((df_filtered.index - start_date).total_seconds() * sample_rate).astype(int)
# 裁剪降雨量数据到台站的时间范围
rain_sample_indices = ((df_filtered.index - start_date).total_seconds() * sample_rate).astype(int)

### 全局事件匹配图

In [ ]:
def plot_combined_global_matched_events_with_rain_5min(stations, offsets, threshold, weight,
													   input_dir='./logs_814-918',
													   output_dir='./analysis_plots/814-918'):
	# 创建输出目录
	os.makedirs(output_dir, exist_ok=True)
	
	# 初始化绘图
	fig = plt.figure(figsize=(50, 10))
	gs = fig.add_gridspec(2, 1, height_ratios=[1, 3])  # 上方为降雨量图，下方为全局匹配事件分布图
	
	# 子图1：降雨量图
	ax_rain = fig.add_subplot(gs[0])
	colors = plt.cm.tab20.colors  # 使用颜色映射区分不同台站
	## 绘制降雨量图，使用 rain_sample_indices 作为 x 轴
	ax_rain.plot(rain_sample_indices, df_filtered['Rainfall(mm)'], label='Rainfall', color='blue')
	ax_rain.set_xlabel('Sample Index (since 2023-08-14)')
	ax_rain.set_ylabel('Rainfall (mm)')
	ax_rain.tick_params(axis='y')
	ax_rain.legend(loc='upper right')
	ax_rain.grid(True)
	ax_rain.set_title("Rainfall Over Time (Sample Index)")
	
	# 子图2：全局匹配事件分布图
	ax_global = fig.add_subplot(gs[1], sharex=ax_rain)
	plot_data = []
	
	for i, (station, offset) in enumerate(zip(stations, offsets)):
		station_dir = os.path.join(input_dir, station)
		result_files = [f for f in os.listdir(station_dir)
						if f.startswith(station + '_matched_filter_results_threshold' + threshold + '_weight' + weight)
						and f.endswith('.h5')]
		all_indices = []
		
		for filename in result_files:
			full_path = os.path.join(station_dir, filename)
			results = utils.load_result(os.path.basename(full_path), path=os.path.dirname(full_path) + '/')
			if not results:
				continue
			
			templates = list(results.keys())
			templates = [t for t in templates if t != 'template_16334774']
			for t in templates:
				all_indices.extend(results[t]['matched_event_index_all'])
		
		# 调整样本索引以匹配降雨量的时间轴
		all_indices = [idx + (3 - offset) * 250 * 60 * 60 * 24 for idx in all_indices]
		all_indices.sort()
		
		# 将 station, offset 和 all_indices 添加到 plot_data 中
		plot_data.append((station, offset, all_indices))
	
	# 根据 all_indices 的长度对 plot_data 进行排序（从多到少）
	plot_data.sort(key=lambda x: len(x[2]))
	
	# 绘制全局匹配事件分布图，按照排序后的顺序分配 y 轴位置
	for i, (station, offset, all_indices) in enumerate(plot_data):
		color = colors[i % len(colors)]
		ax_global.plot(all_indices, [i] * len(all_indices), '|', markersize=10, color=color, label=station)
		#ax_global.plot(select_all_indices, [i] * len(select_all_indices), '|', markersize=10, color=color, label=station)
	ax_global.set_xlabel('Global Event Index: threshold ' + threshold + ', weight ' + weight)
	ax_global.set_yticks(range(len(stations)))
	ax_global.set_yticklabels([station for station, _, _ in plot_data])  # 使用排序后的 station 名称
	ax_global.grid(True)
	ax_global.legend(loc='upper right', bbox_to_anchor=(1.15, 1), fontsize='small')
	
	# 调整布局
	plt.tight_layout()
	#plt.savefig(os.path.join(output_dir, 'combined_global_matched_events_' + threshold + '_' + weight + '.png'))
	plt.savefig(os.path.join(output_dir, 'combined_global_matched_events_' + threshold + '_' + weight +'without_template_16334774'+ '.png'))
	plt.close()

In [ ]:
# 调用函数
stations = ['Xs01', 'Xs02', 'Xs03', 'Xs04', 'Xs05', 'Xs06', 'Xs07', 'Xs08',
			'Xs09', 'Xs10', 'Xs11', 'Xs12', 'Xs13', 'Xs14', 'Xs16', 'Xs17',
			'Xs18', 'Xs19', 'Xs20', 'Xs21', 'Xs22', 'Xs23', 'Xs24', 'Xs25',
			'Xs26', 'Xs27', 'Xs28', 'Xs29']
offsets = [1, 1, 1, 1, 1, 0, 0, 0,
		   0, 0, 0, 0, 0, 2, 2, 2,
		   2, 2, 2, 2, 2, 2, 3, 3,
		   3, 3, 3, 3]
#thresholds = ['0.7','0.8']
#weights = ['1.0_0.0_0.0','0.0_1.0_0.0','0.0_0.0_1.0']
thresholds = ['0.7','0.8']
weights = ['1.0_0.0_0.0']
for threshold in thresholds:
	for weight in weights:
		plot_combined_global_matched_events_with_rain_5min(stations, offsets, threshold, weight)

### 匹配数量图（5min）

In [7]:
def plot_rainfall_and_event_count_per_station(stations, offsets, threshold, weight,
											  input_dir='./logs_814-918',
											  output_dir='./analysis_plots/814-918'):    
	# 遍历每个台站
	for station, offset in zip(stations, offsets):
		print(f"Processing station: {station}")
		
		# 初始化绘图
		fig = plt.figure(figsize=(20, 10))
		gs = fig.add_gridspec(2, 1, height_ratios=[1, 1])  # 上方为降雨量图，下方为柱状图
		
		# 子图1：降雨量图
		ax_rain = fig.add_subplot(gs[0])
		ax_rain.plot(rain_sample_indices, df_filtered['Rainfall(mm)'], label='Rainfall', color='blue')
		ax_rain.set_xlabel('Sample Index (since 2023-08-14)')
		ax_rain.set_ylabel('Rainfall (mm)')
		ax_rain.tick_params(axis='y')
		ax_rain.legend(loc='upper right')
		ax_rain.grid(True)
		ax_rain.set_title(f"Rainfall Over Time for Station {station}")
		
		# 子图2：5分钟事件计数柱状图
		ax_bar = fig.add_subplot(gs[1], sharex=ax_rain)
		
		# 加载该台站的匹配事件数据
		station_dir = os.path.join(input_dir, station)
		result_files = [f for f in os.listdir(station_dir)
						if f.startswith(station + '_matched_filter_results_threshold' + threshold + '_weight' + weight)
						and f.endswith('.h5')]
		all_event_indices = []
		print(result_files)
		for filename in result_files:
			full_path = os.path.join(station_dir, filename)
			results = utils.load_result(os.path.basename(full_path), path=os.path.dirname(full_path) + '/')
			if not results:
				continue
			
			templates = list(results.keys())
			templates = [t for t in templates if t != 'template_16334774']
			for t in templates:
				all_event_indices.extend(results[t]['matched_event_index_all'])	
			# 调整样本索引以匹配降雨量的时间轴
			
			all_event_indices = [idx + (3 - offset) * 250 * 60 * 60 * 24 for idx in all_event_indices]
			all_event_indices.sort()
      
			# 将事件索引转换为时间戳
			event_start_date_offset = pd.Timedelta(days=(3 - offset))  # 匹配事件的起始时间偏移量
			event_start_date = start_date + event_start_date_offset
			event_timestamps = event_start_date + pd.to_timedelta(np.array(all_event_indices) / sample_rate, unit='s')
			
			# 将时间戳划分为5分钟的时间段
			bins = pd.date_range(start=start_date, end=end_date, freq='5T')  # 5分钟间隔
			event_counts, _ = np.histogram(event_timestamps, bins=bins)
			
			# 转换时间戳到样本索引
			bin_sample_indices = ((bins[:-1] - start_date).total_seconds() * sample_rate).astype(int)
			
			# 绘制柱状图
			ax_bar.bar(bin_sample_indices, event_counts, width=np.diff(bin_sample_indices)[0], align='edge', color='orange', alpha=0.7)
			ax_bar.set_xlabel('Sample Index (since 2023-08-14)')
			ax_bar.set_ylabel('Event Count (5 min)')
			ax_bar.grid(True)
			ax_bar.set_title(f"Event Count per 5 Minutes for Station {station}")
			
			# 调整布局
			plt.tight_layout()
			
			# 按照特定目录结构保存图像
			station_output_dir = os.path.join(output_dir, station)
			station_output_dir = os.path.join(station_output_dir, os.path.splitext(filename)[0])
			file_name = f'{station}_matched_filter_results_threshold{threshold}_weight{weight}_5min_matched_result_without_template_16334774.png'
			print(os.path.join(station_output_dir, file_name))
			plt.savefig(os.path.join(station_output_dir, file_name))
			plt.close()


In [8]:
# 调用函数
stations = ['Xs01', 'Xs02', 'Xs03', 'Xs04', 'Xs05', 'Xs06', 'Xs07', 'Xs08',
			'Xs09', 'Xs10', 'Xs11', 'Xs12', 'Xs13', 'Xs14', 'Xs16', 'Xs17',
			'Xs18', 'Xs19', 'Xs20', 'Xs21', 'Xs22', 'Xs23', 'Xs24', 'Xs25',
			'Xs26', 'Xs27', 'Xs28', 'Xs29']
offsets = [1, 1, 1, 1, 1, 0, 0, 0,
		   0, 0, 0, 0, 0, 2, 2, 2,
		   2, 2, 2, 2, 2, 2, 3, 3,
		   3, 3, 3, 3]
#thresholds = ['0.8']
#weights = ['1.0_0.0_0.0','0.0_1.0_0.0','0.0_0.0_1.0']
thresholds = ['0.7','0.8']
weights = ['1.0_0.0_0.0']
# 调用函数
for threshold in thresholds:
	for weight in weights:
		plot_rainfall_and_event_count_per_station(stations, offsets, threshold, weight)

Processing station: Xs01
['Xs01_matched_filter_results_threshold0.7_weight1.0_0.0_0.0.h5']
./analysis_plots/814-918/Xs01/Xs01_matched_filter_results_threshold0.7_weight1.0_0.0_0.0/Xs01_matched_filter_results_threshold0.7_weight1.0_0.0_0.0_5min_matched_result_without_template_16334774.png
Processing station: Xs02
['Xs02_matched_filter_results_threshold0.7_weight1.0_0.0_0.0.h5']
./analysis_plots/814-918/Xs02/Xs02_matched_filter_results_threshold0.7_weight1.0_0.0_0.0/Xs02_matched_filter_results_threshold0.7_weight1.0_0.0_0.0_5min_matched_result_without_template_16334774.png
Processing station: Xs03
['Xs03_matched_filter_results_threshold0.7_weight1.0_0.0_0.0.h5']
./analysis_plots/814-918/Xs03/Xs03_matched_filter_results_threshold0.7_weight1.0_0.0_0.0/Xs03_matched_filter_results_threshold0.7_weight1.0_0.0_0.0_5min_matched_result_without_template_16334774.png
Processing station: Xs04
['Xs04_matched_filter_results_threshold0.7_weight1.0_0.0_0.0.h5']
./analysis_plots/814-918/Xs04/Xs04_matched